In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"
    ) \
    .getOrCreate()

In [ ]:
# Load User CSV
users_df = spark.read.csv(
    "data/user_table.csv",
    header=True,
    inferSchema=True)

In [ ]:
# Transaction Schema
tx_schema = StructType([
    StructField("tx_id", IntegerType(), True),
    StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", DoubleType(), True)
])


In [ ]:
# Read Kafka Stream
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud-detection") \
    .load()

In [ ]:
# Parse JSON
parsed_stream = kafka_stream.select(
    from_json(
        col("value").cast("string"),
        tx_schema
    ).alias("tx")
).select("tx.*")

In [ ]:
# Filter Transactions Above ₹500
high_value_stream = parsed_stream.filter(
    col("amount") > 500
)

In [ ]:
# # Fraud Detection
# fraud_stream = parsed_stream.filter(
#     col("amount") > 10000.0
# )
# # Join with User Data
# enriched_fraud = fraud_stream.join(
#     users_df,
#     "userId"
# )
# Convert back to JSON
# output_stream = enriched_fraud \
#     .withColumn(
#         "value",
#         to_json(struct("*"))
#     ) \
#     .select("value")

# # Write to Kafka
# query = output_stream.writeStream \
#     .format("kafka") \
#     .option("kafka.bootstrap.servers", "kafka:9092") \
#     .option("topic", "fraud-notification") \
#     .option("checkpointLocation", "/workspace/checkpoints") \
#     .start()

# query.awaitTermination()

In [ ]:
# Join with User Data
enriched_stream = high_value_stream.join(
    users_df,
    "userId"
)

In [ ]:
# Convert to JSON
output_stream = enriched_stream \
    .withColumn(
        "value",
        to_json(struct("*"))
    ) \
    .select("value")

# Write to Kafka
query = output_stream.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "transaction-above-500") \
    .option("checkpointLocation", "/workspace/checkpoints_above500") \
    .start()